# Normalize Article Text

This notebook prepares Khmer article text for search. It applies Unicode and whitespace normalization only. It does not translate, correct legal wording, tokenize, segment, or generate embeddings.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'search' / 'articles.jsonl'
OUTPUT_PATH = DATA_DIR / 'processed' / 'search' / 'articles_normalized.jsonl'
ZERO_WIDTH = '\u200b\u200c\u200d\ufeff'
print('Input:', INPUT_PATH)
print('Output:', OUTPUT_PATH)

Input: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/articles.jsonl
Output: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/articles_normalized.jsonl


In [2]:
def normalize_khmer(value):
    text = unicodedata.normalize('NFC', str(value or ''))
    text = text.translate(str.maketrans('', '', ZERO_WIDTH))
    text = text.replace('\u00a0', ' ')
    lines = []
    for line in text.splitlines():
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if line:
            lines.append(line)
    return re.sub(r'\n{2,}', '\n', '\n'.join(lines))


In [3]:
records = []
for line in INPUT_PATH.read_text(encoding='utf-8').splitlines():
    record = json.loads(line)
    normalized = dict(record)
    normalized['search_text'] = normalize_khmer(record['text'])
    normalized['normalization_version'] = 'unicode-nfc-v1'
    records.append(normalized)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for record in records:
        output.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Normalized articles: {len(records):,}')
print(f'Output: {OUTPUT_PATH}')

Normalized articles: 2,744
Output: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/articles_normalized.jsonl


In [4]:
assert records
assert len(records) == len({record['id'] for record in records})
assert all(record['search_text'].strip() for record in records)
assert not any(any(character in record['search_text'] for character in ZERO_WIDTH) for record in records)
print('Validation passed')
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Validation passed
{
  "id": "1-1",
  "article_id": "1-1",
  "document_id": "1",
  "article_number": 1,
  "text": "មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ",
  "document_title": "ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)",
  "document_number": "នស/រកម/០១១៧/០០២",
  "document_type": "Law",
  "source_url": "https://library.ncdd.gov.kh/detail/1",
  "pdf_url": "https://library.ncdd.gov.kh/detail/1/download",
  "search_text": "មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ",
  "normalization_version": "unicode-nfc-v1"
}
